# Librerias necesarias: 

Al trabajar con una página estática del gobierno, es recomendable trabajar con las librerías de request y beautifulsoup, para hacer el árbol html de dicha página (parsear el documento a dicha estructura)

In [1]:
import pandas as pd 
import numpy as np
from bs4 import BeautifulSoup
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import openpyxl
import re
import unicodedata


# Extracción datos

In [18]:
url="https://www.supersolidaria.gov.co/es/entidad/cooperativas-de-ahorro-y-credito"
response = requests.get(url, verify=False)
soup = BeautifulSoup(response.text, 'html.parser')

c:\Users\Liana\AppData\Local\miniconda3\envs\camel\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Organizar los links de la SuperSolidaria para cada año

In [19]:
etiqueta="a"
documentos = soup.find_all(etiqueta, href=lambda href: href and (href.endswith(".xlsx") or href.endswith(".xls") ) )
links_descarga=[documentos[i]["href"] for i in range(len(documentos))]

In [20]:
def nombre(ruta):
    nombre_archivo = ruta.rsplit("/", 1)[-1] if "/" in ruta else "archivo_descargado.xlsx"
    return nombre_archivo.strip("\"'")

def Filtrar_datos(list_links):
    Links_descarga_modif = list(map(lambda ruta: nombre(ruta).split("_"), list_links))
    indices_a_conservar = [i for i in range(len(Links_descarga_modif)) 
                        if ("estados" in Links_descarga_modif[i] and "financieros" in Links_descarga_modif[i]) or 
                        ("financiera" in Links_descarga_modif[i] and "puc" in Links_descarga_modif[i]) or 
                        ("financiera" in Links_descarga_modif[i] and "f" in Links_descarga_modif[i]) or 
                        ("Estados" in Links_descarga_modif[i] and "Financieros" in Links_descarga_modif[i]) or 
                        ("estados" in Links_descarga_modif[i] and "%20financieros" in Links_descarga_modif[i]) or
                        ("Estados" in Links_descarga_modif[i] and "%20Financieros" in Links_descarga_modif[i]) or
                        ("20231117estados" in Links_descarga_modif[i] and "%20financieros" in Links_descarga_modif[i]) 
                        ]
    #20231117estados_ financieros_ahorro_credito_sept_2023.xlsx
    links_descarga_filtrados = [list_links[i] for i in indices_a_conservar]
    return links_descarga_filtrados

def descargar_archivo(url_base, link, carpeta_destino="../tablas/data_historico"):
    url_excel_remote = url_base + link
    try:
        respuesta = requests.get(url_excel_remote, timeout=30, verify=False)
        respuesta.raise_for_status()
        nombre_archivo = nombre(url_excel_remote)
        ruta_completa = os.path.join(carpeta_destino, nombre_archivo)

        with open(ruta_completa, "wb") as f:
            f.write(respuesta.content)
        return #f"{nombre_archivo} descargado correctamente en {carpeta_destino}"
    except Exception as e:
        return f"Error descargando {url_excel_remote}: {e}"

def descargar_Archivos(lista_links, carpeta_destino="../tablas/data_historico"):
    url_base = "https://www.supersolidaria.gov.co"
    resultados = []

    # Crea la carpeta si no existe
    os.makedirs(carpeta_destino, exist_ok=True)

    with ThreadPoolExecutor(max_workers=10) as executor:
        tareas = [
            executor.submit(descargar_archivo, url_base, link, carpeta_destino)
            for link in lista_links
        ]
        for future in as_completed(tareas):
            resultados.append(future.result())

    for r in resultados:
        print(r)



In [21]:
datos_filtrados=Filtrar_datos(list_links=links_descarga)
descargar_Archivos(lista_links=datos_filtrados)

c:\Users\Liana\AppData\Local\miniconda3\envs\camel\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Liana\AppData\Local\miniconda3\envs\camel\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Liana\AppData\Local\miniconda3\envs\camel\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


In [2]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np


# UTILIDADES TEXTO

def sin_tildes(texto):

    return ''.join(
        c for c in unicodedata.normalize('NFD', str(texto))
        if unicodedata.category(c) != 'Mn'
    )


# =========================================================
# EXTRAER MES Y AÑO
# =========================================================

def extraer_mes_ano(path):

    # Leer primeras filas
    preview = pd.read_excel(path, header=None, nrows=10)

    texto = " ".join(
        preview.astype(str)
        .fillna("")
        .values
        .flatten()
        .tolist()
    ).lower()

    texto = sin_tildes(texto)

    meses = [
        'enero', 'febrero', 'marzo', 'abril',
        'mayo', 'junio', 'julio', 'agosto',
        'septiembre', 'octubre', 'noviembre',
        'diciembre'
    ]

    mes = next((m for m in meses if m in texto), None)

    anos = re.findall(r'\b\d{4}\b', texto)

    ano = anos[0] if anos else None

    return mes, ano


# =========================================================
# DETECTAR FILAS IMPORTANTES
# =========================================================

def detectar_filas(raw):

    fila_header = None
    fila_inicio = None

    for i in range(len(raw)):

        valor = raw.iloc[i, 0]

        valor_str = str(valor).strip().upper()

        # Fila encabezado
        if valor_str == "CUENTA":
            fila_header = i

        # Inicio datos
        try:
            if int(float(valor)) == 100000:
                fila_inicio = i
                break
        except:
            pass

    return fila_header, fila_inicio


# =========================================================
# LIMPIAR NOMBRES COLUMNAS
# =========================================================

def limpiar_columnas(columnas):

    nuevas = []

    for col in columnas:

        if pd.isna(col):

            nuevas.append("SIN_NOMBRE")

        else:

            col = (
                str(col)
                .strip()
                .replace("\n", " ")
            )

            nuevas.append(col)

    return nuevas


# =========================================================
# ELIMINAR COLUMNAS BASURA
# =========================================================

def eliminar_columnas_basura(df):

    patrones = [
        "CODIGO ENTIDAD",
        "NIT",
        "SIN_NOMBRE"
    ]

    cols_drop = [
        col for col in df.columns
        if any(
            p in str(col).upper()
            for p in patrones
        )
    ]

    return df.drop(columns=cols_drop, errors="ignore")


# =========================================================
# LEER EXCEL
# =========================================================

def leer_excel(path):

    # =====================================================
    # LEER EXCEL COMPLETO
    # =====================================================

    raw = pd.read_excel(path, header=None)

    # =====================================================
    # BUSCAR FILAS CLAVE
    # =====================================================

    fila_header = None
    fila_inicio = None

    for i in range(len(raw)):

        valor = str(raw.iloc[i, 0]).strip().upper()

        # Fila encabezado
        if valor == "CUENTA":
            fila_header = i

        # Inicio datos
        try:
            if int(float(raw.iloc[i, 0])) == 100000:
                fila_inicio = i
                break
        except:
            pass

    if fila_header is None:
        raise ValueError(
            f"No se encontró fila CUENTA:\n{path}"
        )

    if fila_inicio is None:
        raise ValueError(
            f"No se encontró fila 100000:\n{path}"
        )

    # =====================================================
    # FILA BASE DE ENCABEZADOS
    # =====================================================

    header_actual = raw.iloc[fila_header].tolist()

    # Fila superior
    if fila_header > 0:
        header_superior = raw.iloc[fila_header - 1].tolist()
    else:
        header_superior = [None] * len(header_actual)

    # =====================================================
    # CONSTRUIR NOMBRES COLUMNAS
    # =====================================================

    columnas = []

    for idx, col in enumerate(header_actual):

        actual = str(col).strip()

        superior = str(header_superior[idx]).strip()

        # -----------------------------------------
        # Primeras columnas fijas
        # -----------------------------------------

        if idx == 0:
            columnas.append("CUENTA")
            continue

        if idx == 1:
            columnas.append("NOMBRE CUENTA")
            continue

        # -----------------------------------------
        # Omitir columnas basura
        # -----------------------------------------

        texto_union = f"{superior} {actual}".upper()
        if actual in ('nan', '') and superior in ('nan', ''):
            columnas.append(None)
            continue

        if any(
            basura in texto_union
            for basura in [
                "NOMBRE ENTIDAD",
                "CODIGO ENTIDAD",
                "NIT"
            ]
        ):
            columnas.append(None)
            continue

        # -----------------------------------------
        # Si actual NO es cooperativa
        # usar superior
        # -----------------------------------------

        _CODIGO_RE = re.compile(r'^E?\d+$')  # definir una vez fuera del loop

        # dentro del loop:
        if _CODIGO_RE.match(actual):
            # actual es un código (E374, 374, E1119, etc.) → siempre usar el nombre de la fila superior
            columnas.append(superior if superior not in ('nan', '') else actual)
        else:
            # actual ya es un nombre → usarlo directamente
            columnas.append(actual)

    # =====================================================
    # EXTRAER DATOS
    # =====================================================

    df = raw.iloc[fila_inicio:].copy()

    # Eliminar duplicados: conservar solo la primera aparición
    vistos = set()
    columnas_dedup = []
    for col in columnas:
        if col is None:
            columnas_dedup.append(None)
        elif col in vistos:
            columnas_dedup.append(None)  # marcar como None para que se elimine después
        else:
            vistos.add(col)
            columnas_dedup.append(col)

    columnas = columnas_dedup

    # Asignar columnas
    df.columns = columnas

    # Eliminar columnas None
    df = df.loc[:, df.columns.notna()]

    # =====================================================
    # LIMPIAR NOMBRES
    # =====================================================

    df.columns = [
        str(col)
        .replace("\n", " ")
        .replace("  ", " ")
        .strip()
        for col in df.columns
    ]

    # =====================================================
    # LIMPIAR FILAS
    # =====================================================

    df = clean_rows(df)

    return df.reset_index(drop=True)


# =========================================================
# LIMPIAR FILAS
# =========================================================

def clean_rows(df):

    df = df.copy()

    # Eliminar filas vacías
    df = df[df["CUENTA"].notna()]

    df = df[df["NOMBRE CUENTA"].notna()]

    # Convertir cuenta
    df["CUENTA"] = pd.to_numeric(
        df["CUENTA"],
        errors="coerce"
    )

    # Mantener cuentas válidas
    df = df[df["CUENTA"].notna()]

    # =====================================================
    # CONVERTIR COLUMNAS NUMÉRICAS
    # =====================================================

    columnas_excluir = [
        "CUENTA",
        "NOMBRE CUENTA",
        "MES",
        "AÑO"
    ]

    columnas_valores = [
        col for col in df.columns
        if col not in columnas_excluir
    ]

    # asegurarse de que los nombres son strings planos
    df.columns = [
        str(col)
        if not isinstance(col, tuple)
        else " ".join(map(str, col))
        for col in df.columns
    ]

    df[columnas_valores] = (
        df[columnas_valores]
        .apply(pd.to_numeric, errors="coerce")
    )

    # =====================================================
    # LLENAR NaN CON 0
    # =====================================================

    df[columnas_valores] = (
        df[columnas_valores]
        .fillna(0)
    )

    return df


# =========================================================
# LEER MUCHOS EXCELS
# =========================================================

def leer_excels(list_name_excel, carpeta):

    lista_archivos = []

    for nombre in list_name_excel:

        path = os.path.join(carpeta, nombre)
        try:

            df = leer_excel(path)
            print("duplicated: ", df.columns[df.columns.duplicated()])

            mes, ano = extraer_mes_ano(path)

            df = df.assign(MES=mes, AÑO=ano)

            lista_archivos.append(df)

        except Exception as e:

            print(f"\nError leyendo {nombre}")
            print(e)

    return lista_archivos

In [3]:
carpeta = '../../tablas/data_historico'
nombres_archivos = [f for f in os.listdir(carpeta) if os.path.isfile(os.path.join(carpeta, f))]

In [4]:
list_excel=leer_excels(list_name_excel=nombres_archivos, carpeta=carpeta)
list_excel[0]

duplicated:  Index([], dtype='str')


C:\Users\Liana\AppData\Local\Temp\ipykernel_21144\3863667932.py:377: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(MES=mes, AÑO=ano)


duplicated:  Index([], dtype='str')


C:\Users\Liana\AppData\Local\Temp\ipykernel_21144\3863667932.py:377: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(MES=mes, AÑO=ano)


duplicated:  Index([], dtype='str')


...

duplicated:  Index([], dtype='str')


C:\Users\Liana\AppData\Local\Temp\ipykernel_21144\3863667932.py:377: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(MES=mes, AÑO=ano)


duplicated:  Index([], dtype='str')


C:\Users\Liana\AppData\Local\Temp\ipykernel_21144\3863667932.py:377: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(MES=mes, AÑO=ano)


,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,...,COOPERATIVA DE AHORRO Y CREDITO COLANTA,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO CAJA UNION COOPERATIVA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA,COOPERATIVA SUYA,MES,AÑO
0,100000,ACTIVO,1.453604e+11,1.204461e+10,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,...,317470642750,1.964690e+11,1.019668e+10,7.498950e+09,8.204825e+10,7.673873e+10,3.689148e+10,6.785967e+10,febrero,2022
1,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,1.094558e+10,1.108710e+09,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,...,27649449122,1.904275e+10,9.197822e+08,5.507006e+08,9.982434e+09,2.782317e+09,5.564935e+09,1.106163e+10,febrero,2022
2,110500,CAJA,2.868046e+08,1.061263e+07,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,...,5735744815,2.193096e+09,6.007630e+07,4.744467e+08,4.490329e+08,2.018711e+08,1.615997e+08,1.612913e+09,febrero,2022
3,110505,CAJA GENERAL,2.791246e+08,1.061263e+07,8.180852e+08,0.000000e+00,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,...,5732194815,2.182646e+09,5.907630e+07,4.738467e+08,4.450329e+08,1.998711e+08,1.600997e+08,1.612913e+09,febrero,2022
4,110510,CAJA MENOR,7.680000e+06,0.000000e+00,4.850000e+06,1.100000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,...,3550000,1.045000e+07,1.000000e+06,6.000000e+05,4.000000e+06,2.000000e+06,1.500000e+06,0.000000e+00,febrero,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2082,931500,BIENES Y VALORES RECIBIDOS EN ADMINISTRACIÓN,0.000000e+00,0.000000e+00,2.470562e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,febrero,2022
2083,931505,CARTERA FOGACOOP,0.000000e+00,0.000000e+00,2.470562e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,febrero,2022
2084,960000,ACREEDORAS POR CONTRA (DB),1.136827e+10,1.132251e+10,4.492767e+11,2.265980e+11,1.485778e+11,6.602588e+09,6.171632e+10,2.998245e+10,...,366282615637,1.432267e+11,1.630302e+10,7.306368e+09,1.595570e+10,7.249165e+08,0.000000e+00,1.997420e+10,febrero,2022
2085,960500,RESPONSABILIDADES CONTINGENTES POR EL CONTRARIO,1.136827e+10,1.132251e+10,4.492767e+11,2.265980e+11,1.485778e+11,6.602588e+09,6.171632e+10,2.998245e+10,...,366282615637,1.432267e+11,1.630302e+10,7.306368e+09,1.595570e+10,7.249165e+08,0.000000e+00,1.997420e+10,febrero,2022


En principio son 287 cooperativas

In [5]:
Datos_2013_2025_cooperativas = pd.concat(list_excel, ignore_index=True)

In [6]:
del Datos_2013_2025_cooperativas['nan']

In [7]:
nombres_col = Datos_2013_2025_cooperativas.columns.to_list()
with open('nombres_cooperativas_iniciales.txt', 'w') as file:
    # Join elements with a newline character and write
    file.write('\n'.join(nombres_col))

Se deben eliminar cooperativas que tengan muchos valores vacíos, por ejemplo 'COOPERATIVA AVANZA'

In [ ]:
# Ejemplo de que los datos de algunas cooperativas están completos
Datos_2013_2025_cooperativas['COOPANTEX COOPERATIVA DE AHORRO Y CREDITO'].isna().sum()

np.int64(0)

In [8]:
# Diccionario para convertir nombres de mes a número
meses_es = {
    'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4,
    'mayo': 5, 'junio': 6, 'julio': 7, 'agosto': 8,
    'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12
}

# Convertir nombre del mes a número
Datos_2013_2025_cooperativas['mes_num'] = Datos_2013_2025_cooperativas['MES'].str.lower().map(meses_es)


Datos_2013_2025_cooperativas['fecha'] = pd.to_datetime(dict(year=Datos_2013_2025_cooperativas['AÑO'],
                                                                month=Datos_2013_2025_cooperativas['mes_num'], day=1))
Datos_2013_2025_cooperativas.drop(["MES", "AÑO", "mes_num"], inplace=True,axis=1)
Datos_2013_2025_cooperativas.head()

C:\Users\Liana\AppData\Local\Temp\ipykernel_21144\500060389.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Datos_2013_2025_cooperativas['mes_num'] = Datos_2013_2025_cooperativas['MES'].str.lower().map(meses_es)
C:\Users\Liana\AppData\Local\Temp\ipykernel_21144\500060389.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Datos_2013_2025_cooperativas['fecha'] = pd.to_datetime(dict(year=Datos_2013_2025_cooperativas['AÑO'],


,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,...,COOPERATIVA DE AHORRO Y CREDITO INTEGRAR,COOPERATIVA MULTIACTIVA DE APORTE Y CREDITO,MICROEMPRESAS DE ANTIOQUIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,COOPERATIVA MULTIACTIVA UNION DE COMERCIANTES PLAZA TRINIDAD GALAN,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA AVANZA,COOPERATIVA MULTIACTIVA UNION COLOMBIANA,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,fecha
0,100000,ACTIVO,1.453604e+11,1.204461e+10,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
1,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,1.094558e+10,1.108710e+09,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
2,110500,CAJA,2.868046e+08,1.061263e+07,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
3,110505,CAJA GENERAL,2.791246e+08,1.061263e+07,8.180852e+08,0.000000e+00,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
4,110510,CAJA MENOR,7.680000e+06,0.000000e+00,4.850000e+06,1.100000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01


In [9]:
#eliminar datos que no parezcan en todos los años
anos_unicos = Datos_2013_2025_cooperativas['fecha'].dt.year.unique() 
for ano in anos_unicos:
    data_ano = Datos_2013_2025_cooperativas[Datos_2013_2025_cooperativas['fecha'].dt.year == ano]
    columnas_datos = data_ano.drop(columns='fecha')
    columnas_a_eliminar = columnas_datos.columns[(columnas_datos == 0).all()]
    Datos_2013_2025_cooperativas.drop(columns=columnas_a_eliminar, inplace=True)

In [10]:
Datos_2013_2025_cooperativas

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,...,COOPERATIVA DE AHORRO Y CREDITO INTEGRAR,COOPERATIVA MULTIACTIVA DE APORTE Y CREDITO,MICROEMPRESAS DE ANTIOQUIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,COOPERATIVA MULTIACTIVA UNION DE COMERCIANTES PLAZA TRINIDAD GALAN,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA AVANZA,COOPERATIVA MULTIACTIVA UNION COLOMBIANA,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,fecha
0,100000,ACTIVO,1.453604e+11,1.204461e+10,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
1,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,1.094558e+10,1.108710e+09,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
2,110500,CAJA,2.868046e+08,1.061263e+07,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
3,110505,CAJA GENERAL,2.791246e+08,1.061263e+07,8.180852e+08,0.000000e+00,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
4,110510,CAJA MENOR,7.680000e+06,0.000000e+00,4.850000e+06,1.100000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320597,935000,OTRAS ACREEDORAS DE CONTROL,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.900000e+01,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.0,2014-02-01
320598,960000,ACREEDORAS CONTINGENTES POR CONTRA,NaN,7.286149e+09,2.645291e+11,1.493243e+08,5.438012e+10,2.747754e+09,4.058720e+10,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,6.970617e+10,2.208899e+10,34000000.0,2014-02-01
320599,960500,ACREEDORAS CONTINGENTES POR CONTRA (DB),NaN,7.286149e+09,2.645291e+11,1.493243e+08,5.438012e+10,2.747754e+09,4.058720e+10,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,6.970617e+10,2.208899e+10,34000000.0,2014-02-01
320600,980000,ACREEDORAS DE CONTROL POR CONTRA,NaN,1.274480e+09,2.619920e+10,3.159206e+11,7.700000e+09,1.299760e+09,6.160000e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4.596689e+09,2.710400e+09,0.0,2014-02-01


In [11]:
Datos_2013_2025_cooperativas[Datos_2013_2025_cooperativas["CUENTA"] == 141015]

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,...,COOPERATIVA DE AHORRO Y CREDITO INTEGRAR,COOPERATIVA MULTIACTIVA DE APORTE Y CREDITO,MICROEMPRESAS DE ANTIOQUIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,COOPERATIVA MULTIACTIVA UNION DE COMERCIANTES PLAZA TRINIDAD GALAN,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA AVANZA,COOPERATIVA MULTIACTIVA UNION COLOMBIANA,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,fecha
293,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
2379,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
4465,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-01-01
6551,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-04-01
8643,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-05-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256264,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019-05-01
258310,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-05-01
260397,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-05-01
262484,141015,CATEGORÍA C RIESGO APRECIABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-05-01


Se exporta a csv porque es muy grande para exportar a excel

In [13]:
Datos_2013_2025_cooperativas.to_csv("../../tablas/Datos_2013_2025_cooperativas.csv", index=False)

# Consideraciones de los datos

* Filtrar datos de las cooperativas del 2021 en adelante (porque hemos revisado la disponibilidad de los valores de irl y solvencia de la SuperIntendencia y están desde ese año)
* El alto porcentaje de ceros no es necesariamente datos faltantes, es estructural. Una cooperativa peuqña tendrá cero en la mayoría de cuentas simplemente porque no opera en esos rubros. Por ello, el criterio para eliminar cooperativas no debe basarse solo en 'un montón de ceros', sino en los nulos, y además evaluarse a nivel de cuentas clave para CAMEL (que se revisa en el otro notebook)

In [11]:
datos_filtrados_fecha = Datos_2013_2025_cooperativas[Datos_2013_2025_cooperativas["fecha"] >= "2021-01-01"]
datos_filtrados_fecha.head()

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,...,COOPERATIVA DE AHORRO Y CREDITO INTEGRAR,COOPERATIVA MULTIACTIVA DE APORTE Y CREDITO,MICROEMPRESAS DE ANTIOQUIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,COOPERATIVA MULTIACTIVA UNION DE COMERCIANTES PLAZA TRINIDAD GALAN,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA AVANZA,COOPERATIVA MULTIACTIVA UNION COLOMBIANA,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,fecha
0,100000,ACTIVO,1.453604e+11,1.204461e+10,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
1,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,1.094558e+10,1.108710e+09,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
2,110500,CAJA,2.868046e+08,1.061263e+07,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
3,110505,CAJA GENERAL,2.791246e+08,1.061263e+07,8.180852e+08,0.000000e+00,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
4,110510,CAJA MENOR,7.680000e+06,0.000000e+00,4.850000e+06,1.100000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01


In [15]:
# Eliminar columnas con más de 1 valor nulo
nulos_por_columna = datos_filtrados_fecha.isna().sum()
columnas_a_eliminar = nulos_por_columna[nulos_por_columna > 1].index
datos_filtrados_fecha = datos_filtrados_fecha.drop(columns=columnas_a_eliminar)
print(f"Se eliminaron {len(columnas_a_eliminar)} columnas")
print(f"Columnas eliminadas: {list(columnas_a_eliminar)}")
print(f"Columnas restantes: {len(datos_filtrados_fecha.columns)}")

Se eliminaron 147 columnas
Columnas eliminadas: ['COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR', 'COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL', 'PROGRESSA ENTIDAD COOPERATIVA DE AHORRO Y CRÉDITO', 'COOPERATIVA DE AHORRO Y CREDITO DE CHIPAQUE', 'COOPERATIVA NACIONAL DE TRABAJADORES DE LA INDUSTRIA DE LAS GASEOSAS Y BEBIDAS', 'COOPERATIVA DE AHORRO Y CREDITO LIMITADA', 'COOPERATIVA MULTIACTIVA EMPRESARIAL COOVITEL', 'COPERATIVA INDEPENDIENTE DE EMPLEADOS DE ANTIOQUIA', 'COOPERATIVA DE EMPLEADOS DE LA REGISTRADURIA NACIONAL', 'COOPERATIVA DE AHORRO Y CREDITO COTRAMED', 'COOPERATIVA ESPECIALIZADA DE AHORRO Y CRÉDITO ORBISCOOP', 'COOPERATIVA ANTIOQUEÑA DE TRABAJADORES GRUPO CAFETERO', 'COOPERATIVA DE AHORRO Y CREDITO SERVUNAL', 'COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO UNIVERSIDAD DE MEDELLIN', 'COOPERATIVA DE TRABAJADORES PANAMCO COLOMBIA S A MEDELLIN', 'COMFAMIGOS COOPERATIVA MULTIACTIVA', 'COOPERATIVA DE TRABAJADORES DE ENKA LTDA', 'COOPERATIVA MULTIACTIVA SANTA 

In [12]:
datos_filtrados_fecha.to_csv("../../tablas/Datos_2021_2025_cooperativas.csv", index=False)

# **Revisar las cuentas que usan los indicadores que dan cero**


In [12]:
import pandas as pd

data = pd.read_csv("../../tablas/Datos_2013_2025_cooperativas.csv")
data.head(3)

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,...,COOPERATIVA DE AHORRO Y CREDITO INTEGRAR,COOPERATIVA MULTIACTIVA DE APORTE Y CREDITO,MICROEMPRESAS DE ANTIOQUIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,COOPERATIVA MULTIACTIVA UNION DE COMERCIANTES PLAZA TRINIDAD GALAN,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA AVANZA,COOPERATIVA MULTIACTIVA UNION COLOMBIANA,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,fecha
0,100000,ACTIVO,1.453604e+11,1.204461e+10,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
1,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,1.094558e+10,1.108710e+09,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
2,110500,CAJA,2.868046e+08,1.061263e+07,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01


In [15]:
data2 = data[data["fecha"] >= "2020-01-01"]

In [16]:
data2

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,...,COOPERATIVA DE AHORRO Y CREDITO INTEGRAR,COOPERATIVA MULTIACTIVA DE APORTE Y CREDITO,MICROEMPRESAS DE ANTIOQUIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,COOPERATIVA MULTIACTIVA UNION DE COMERCIANTES PLAZA TRINIDAD GALAN,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA AVANZA,COOPERATIVA MULTIACTIVA UNION COLOMBIANA,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,fecha
0,100000,ACTIVO,1.453604e+11,1.204461e+10,3.387150e+11,1.560676e+11,9.879448e+10,1.064674e+10,7.021719e+10,3.751248e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
1,110000,EFECTIVO Y EQUIVALENTE AL EFECTIVO,1.094558e+10,1.108710e+09,1.566943e+10,2.669495e+10,1.918907e+09,1.552585e+09,4.685907e+09,1.926925e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
2,110500,CAJA,2.868046e+08,1.061263e+07,8.229352e+08,1.100000e+06,3.589941e+08,1.718631e+08,1.241549e+08,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
3,110505,CAJA GENERAL,2.791246e+08,1.061263e+07,8.180852e+08,0.000000e+00,3.589941e+08,1.718631e+08,1.146549e+08,0.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
4,110510,CAJA MENOR,7.680000e+06,0.000000e+00,4.850000e+06,1.100000e+06,0.000000e+00,0.000000e+00,9.500000e+06,8.000000e+05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
266315,930500,MERCANCÍAS RECIBIDAS EN CONSIGNACIÓN,2.152890e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.242174e+10,2.808970e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-01
266316,931000,BIENES RECIBIDOS DE TERCEROS,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.808970e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-01
266317,960000,ACREEDORAS POR CONTRA (DB),2.152890e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.242174e+10,0.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-01
266318,960500,RESPONSABILIDADES CONTINGENTES POR EL CONTRARIO,2.101304e+11,1.313273e+10,3.433216e+11,1.152840e+11,1.279170e+11,6.093585e+09,6.572081e+10,3.279142e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-01


In [17]:
data["COOPERATIVA DE AHORRO Y CREDITO INTEGRAR"].isna().sum()

np.int64(281408)

In [18]:
len(data)

320602

In [20]:
# Eliminar columnas con más de 1 valor nulo
nulos_por_columna = data2.isna().sum()
columnas_a_eliminar = nulos_por_columna[nulos_por_columna > 1].index
data = data2.drop(columns=columnas_a_eliminar)
print(f"Se eliminaron {len(columnas_a_eliminar)} columnas")
print(f"Columnas eliminadas: {list(columnas_a_eliminar)}")
print(f"Columnas restantes: {len(data.columns)}")

Se eliminaron 147 columnas
Columnas eliminadas: ['COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR', 'COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL', 'PROGRESSA ENTIDAD COOPERATIVA DE AHORRO Y CRÉDITO', 'COOPERATIVA DE AHORRO Y CREDITO DE CHIPAQUE', 'COOPERATIVA NACIONAL DE TRABAJADORES DE LA INDUSTRIA DE LAS GASEOSAS Y BEBIDAS', 'COOPERATIVA DE AHORRO Y CREDITO LIMITADA', 'COOPERATIVA MULTIACTIVA EMPRESARIAL COOVITEL', 'COPERATIVA INDEPENDIENTE DE EMPLEADOS DE ANTIOQUIA', 'COOPERATIVA DE EMPLEADOS DE LA REGISTRADURIA NACIONAL', 'COOPERATIVA DE AHORRO Y CREDITO COTRAMED', 'COOPERATIVA ESPECIALIZADA DE AHORRO Y CRÉDITO ORBISCOOP', 'COOPERATIVA ANTIOQUEÑA DE TRABAJADORES GRUPO CAFETERO', 'COOPERATIVA DE AHORRO Y CREDITO SERVUNAL', 'COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO UNIVERSIDAD DE MEDELLIN', 'COOPERATIVA DE TRABAJADORES PANAMCO COLOMBIA S A MEDELLIN', 'COMFAMIGOS COOPERATIVA MULTIACTIVA', 'COOPERATIVA DE TRABAJADORES DE ENKA LTDA', 'COOPERATIVA MULTIACTIVA SANTA 

Revisar COOPANTEX:

In [25]:
data_coopantex = pd.read_csv("../../tablas/camel/Registros_CAMEL_sin_solvencia.csv")
data_coopantex = data_coopantex[data_coopantex["ID_cooperativa"] == "COOPANTEX COOPERATIVA DE AHORRO Y CREDITO"]
data_coopantex

,ID_registro,ano,mes,ID_indicador,ID_cooperativa,valor
5118,5479,2021,1,Quebranto Patrimonial,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,1.440600
5119,5481,2021,1,Relación entre Aportes sociales mínimos no red...,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.666482
5120,5482,2021,1,Relación entre el Capital Institucional y el A...,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.044706
5121,5483,2021,2,Quebranto Patrimonial,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,1.415697
5122,5485,2021,2,Relación entre Aportes sociales mínimos no red...,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.636056
...,...,...,...,...,...,...
5966,6387,2025,8,Indicador de Riesgo de Liquidez - IRL,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000
5967,6388,2025,9,Indicador de Riesgo de Liquidez - IRL,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000
5968,6389,2025,10,Indicador de Riesgo de Liquidez - IRL,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000
5969,6390,2025,11,Indicador de Riesgo de Liquidez - IRL,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000


In [27]:
data_coopantex[data_coopantex["ID_indicador"] == "Estructura de Balance"]

,ID_registro,ano,mes,ID_indicador,ID_cooperativa,valor
5563,5984,2021,1,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,1.338180
5567,5988,2021,2,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,1.389044
5571,5992,2021,3,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,1.328757
5575,5996,2021,4,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,1.318198
5579,6000,2021,5,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,1.270906
5583,6004,2021,6,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000
5587,6008,2021,7,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000
5591,6012,2021,8,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000
5595,6016,2021,9,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000
5599,6020,2021,10,Estructura de Balance,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,0.000000


In [22]:
data2["COOPANTEX COOPERATIVA DE AHORRO Y CREDITO"]

0         2.923580e+11
1         2.234593e+10
2         1.111631e+09
3         1.109231e+09
4         2.399999e+06
              ...     
266315    3.950114e+10
266316    0.000000e+00
266317    3.950114e+10
266318    7.661069e+10
266319    7.661069e+10
Name: COOPANTEX COOPERATIVA DE AHORRO Y CREDITO, Length: 160434, dtype: float64

In [21]:
def cuentas_con_ceros(df, cuenta: int):
    df_copy = df[df["CUENTA"] == cuenta]
    ceros = (df_copy == 0).sum()
    print(ceros[ceros > 0])

## Capital:

In [19]:
# Aportes sociales mínimos no reducibles y Capital Social usa la cuenta 311000 y 310000
cuentas_con_ceros(data, 311000)
print("\n")
cuentas_con_ceros(data, 310000)

COOPERATIVA MULTIACTIVA DE EL PAUJIL CAQUETA LIMITADA     2
COOPERATIVA TOLIMENSE DE AHORRO Y CREDITO COOFINANCIAR    1
COOPERATIVA LATINOAMERICANA DE AHORRO Y CREDITO           1
COOPERATIVA DE AHORRO Y CREDITO UNION COLOMBIANA          1
dtype: int64


COOPERATIVA MULTIACTIVA DE EL PAUJIL CAQUETA LIMITADA     2
COOPERATIVA TOLIMENSE DE AHORRO Y CREDITO COOFINANCIAR    1
COOPERATIVA LATINOAMERICANA DE AHORRO Y CREDITO           1
COOPERATIVA DE AHORRO Y CREDITO UNION COLOMBIANA          1
dtype: int64


In [25]:
# Capital Institucional y el Activo Total
cuentas_con_ceros(data, 320000)


COOPERATIVA MULTIACTIVA DE EL PAUJIL CAQUETA LIMITADA      2
COOCERVUNION COOPERATIVA DE AHORRO Y CREDITO              25
COOPERATIVA DE AHORRO Y CREDITO CONGENTE                   1
COOPERATIVA TOLIMENSE DE AHORRO Y CREDITO COOFINANCIAR     1
COOPERATIVA LATINOAMERICANA DE AHORRO Y CREDITO            1
COOPERATIVA DE AHORRO Y CREDITO UNION COLOMBIANA           1
COOPERATIVA DE AHORRO Y CREDITO CAJA UNION COOPERATIVA    25
dtype: int64


In [26]:
cuentas_con_ceros(data, 330000)

COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA                       13
COOPERATIVA AVP                                                37
FEBOR ENTIDAD COOPERATIVA                                      10
COOPERATIVA DE PROFESORES DE LA U NACIONAL DE COLOMBIA          8
COOPERATIVA TEXAS LTDA                                         63
COOPERATIVA SAN PIO X DE GRANADA LTDA                          63
COOPERATIVA DE AHORRO Y CREDITO JUAN DE DIOS GOMEZ             54
COOPERATIVA MULTIACTIVA DE EL PAUJIL CAQUETA LIMITADA           2
COOPERATIVA DE AHORRO Y CREDITO COOYAMOR                        1
COOPERATIVA MULTIACTIVA EL BAGRE LTDA                          12
COOPERATIVA DE AHORRO Y CREDITO SAN LUIS                       24
COOPERATIVA MULTIACTIVA DE EMPLEADOS DE COLGATE PALMOLIVE      60
COOPERATIVA TOLIMENSE DE AHORRO Y CREDITO COOFINANCIAR          1
COOPERATIVA CALDENSE DEL PROFESOR                               6
COOPERATIVA LATINOAMERICANA DE AHORRO Y CREDITO                 1
COOPERATIV

In [ ]:
cuentas_con_ceros(data, 340000)

COOPERATIVA DE EMPLEADOS DE CAFAM                             63
COOPERATIVA PARA EL BIENESTAR SOCIAL                          63
COOPERATIVA FINANCIERA SAN FRANCISCO                          63
COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA                      63
FEBOR ENTIDAD COOPERATIVA                                     63
                                                              ..
MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO     14
COOPERATIVA DE AHORRO Y CREDITO CAJA UNION COOPERATIVA        63
COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO          63
LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO                  63
COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA    63
Length: 86, dtype: int64


In [28]:
cuentas_con_ceros(data, 100000)

COOPERATIVA MULTIACTIVA DE EL PAUJIL CAQUETA LIMITADA     2
COOPERATIVA TOLIMENSE DE AHORRO Y CREDITO COOFINANCIAR    1
COOPERATIVA LATINOAMERICANA DE AHORRO Y CREDITO           1
COOPERATIVA DE AHORRO Y CREDITO UNION COLOMBIANA          1
dtype: int64


## Assets

In [31]:
# ind. calidad por riesgo
cuentas_con_ceros(data, 141115)

COOPERATIVA DE EMPLEADOS DE CAFAM                              36
COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS         49
COOPERATIVA PARA EL BIENESTAR SOCIAL                           55
COOPERATIVA FINANCIERA SAN FRANCISCO                           55
COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA        45
                                                               ..
COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA    55
COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO           55
LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO                   55
COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA     55
COOPERATIVA SUYA                                               55
Length: 140, dtype: int64


El indicador anterior es una suma de varias cuentas (la mayoría tienen esa cantidad de ceros en las cooperativas, o cantidades similares)

In [32]:
# ind. calidad por riesgo con castigos
cuentas_con_ceros(data, 140400)

COOPERATIVA PARA EL BIENESTAR SOCIAL                                             63
COOPERATIVA FINANCIERA SAN FRANCISCO                                             63
COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA                          63
FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS                                  63
COOPERATIVA DE AHORRO Y CREDITO DE TRABAJADORES DE PELDAR Y OTROS DE COLOMBIA    32
                                                                                 ..
COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA                      63
COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO                             10
LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO                                     63
COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA                       63
COOPERATIVA SUYA                                                                 63
Length: 106, dtype: int64


Pasa lo mismo que el indicador de calidad por riesgo

In [33]:
# Indicador de Cobertura de la Cartera Total en Riesgo
cuentas_con_ceros(data, 145100)

COOPERATIVA DE EMPLEADOS DE CAFAM                              63
COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS         63
COOPERATIVA PARA EL BIENESTAR SOCIAL                           63
COOPERATIVA FINANCIERA SAN FRANCISCO                           63
COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA        63
                                                               ..
COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA    63
COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO           63
LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO                   63
COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA     63
COOPERATIVA SUYA                                               63
Length: 140, dtype: int64


In [ ]:
data[(data["CUENTA"] == 145100) & (data["fecha"] == "2024-01-01")]

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,COOPERATIVA AVP,FEBOR ENTIDAD COOPERATIVA,...,COOPERATIVA DE AHORRO Y CREDITO DE DROGUISTAS DETALLISTAS,COOPERATIVA DE AHORRO Y CREDITO COLANTA,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO CAJA UNION COOPERATIVA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA,COOPERATIVA SUYA,fecha
25771,145100,DETERIORO MICROCRÉDITO INMOBILIARIO (CR),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-01-01


## Liquidity

In [47]:
# Activos líquidos ...
cuentas_con_ceros(data, 212510)

COOPERATIVA DE EMPLEADOS DE CAFAM                             63
COOPERATIVA PARA EL BIENESTAR SOCIAL                          63
COOPERATIVA FINANCIERA SAN FRANCISCO                          63
COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA       63
COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA                      63
                                                              ..
COOPERATIVA DE AHORRO Y CREDITO CAJA UNION COOPERATIVA        63
COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO          63
LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO                  63
COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA    63
COOPERATIVA SUYA                                              63
Length: 115, dtype: int64


## Análisis

Consideraciones:

* Como aprox el mayor valor de cantidad de ceros en una cooperativa es de 63. Si el 10% de todos los datos es aprox 13,569 esa cantidad de ceros no afecta en general, pero sí de pronto un año específico, como la celda anterior, que no solo se considera eliminar una cooperativa sino una fecha, **se elimina el año donde esté lleno de ceros?? o que tengan mas de un porcentaje específico??**
* Si hay en total 135,691 filas y 67,361 están llenas de cero, ese valor corresponde casi a la mitad del dataframe, y no se podría eliminar las fechas de forma arbitraria. **Tiene sentido que ciertas cooperativas no reporten esas cuentas??**

In [6]:
columnas = data.columns.drop(["CUENTA", "NOMBRE CUENTA", "fecha"])

df_filtrado = data[(data[columnas] == 0).all(axis=1)]

df_filtrado

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA,COOPERATIVA AVP,FEBOR ENTIDAD COOPERATIVA,...,COOPERATIVA DE AHORRO Y CREDITO DE DROGUISTAS DETALLISTAS,COOPERATIVA DE AHORRO Y CREDITO COLANTA,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO CAJA UNION COOPERATIVA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,COPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,COOPERTAIVA ESPECIALIZADA DE AHORRO Y CREDITO TAX LA FERIA,COOPERATIVA SUYA,fecha
16,112001,FONDO DE LIQUIDEZ - CUENTAS CORRIENTES,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-02-01
24,112020,FIDEICOMISO FONDO DE REPOSICIÓN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-02-01
25,112025,FIDEICOMISO DE ADMINISTRACIÓN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-02-01
26,112030,OTROS FONDOS ESPECIALES,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-02-01
27,112035,FONDOS DE CAMBIO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135630,710500,MANO DE OBRA DIRECTA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-05-01
135631,711000,COSTOS INDIRECTOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-05-01
135632,711500,CONTRATOS DE SERVICIOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-05-01
135633,800000,CUENTAS DE REVELACIÓN DE INFORMACIÓN FINANCIER...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-05-01


In [7]:
df_filtrado["fecha"].unique().tolist()

['2022-02-01',
 '2023-01-01',
 '2023-04-01',
 '2023-05-01',
 '2023-03-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2025-02-01',
 '2025-03-01',
 '2025-01-01',
 '2025-04-01',
 '2025-05-01',
 '2026-01-01',
 '2026-02-01',
 '2026-03-01',
 '2023-02-01',
 '2021-04-01',
 '2022-04-01',
 '2021-05-01',
 '2022-05-01',
 '2021-02-01',
 '2021-01-01',
 '2022-01-01',
 '2021-03-01',
 '2022-03-01']

In [9]:
df_filtrado["fecha"].value_counts().sort_index()

fecha
2021-01-01    2045
2021-02-01    2029
2021-03-01    1012
2021-04-01    1010
2021-05-01    6026
2022-01-01    1041
2022-02-01    2997
2022-03-01    1017
2022-04-01    1012
2022-05-01    5999
2023-01-01    1076
2023-02-01    1055
2023-03-01    3154
2023-04-01    1047
2023-05-01    6362
2024-01-01    1071
2024-02-01    1050
2024-03-01    1083
2024-04-01    1042
2024-05-01    8636
2025-01-01    2163
2025-02-01    1096
2025-03-01    3313
2025-04-01    1085
2025-05-01    6403
2026-01-01    1156
2026-02-01    1197
2026-03-01    1184
Name: count, dtype: int64

**Siempre coincide con que esté llenos de ceros los meses de Enero a Mayo, eso indica algo??**